# Objective
### The primary goal of this project is to develop a highly accurate machine learning capable of predicting the age of an abalone from physical measurementsa

### Currently, determining the age of the abalone involves cutting the shell through the cone, staining it and counting the numbers of rings through the microscope.
### This is a tediuc, resource-intensive and time consuming task. By leveraging easily obtainable physical measurements to predict the number of ring (Where age is years equals rings +1.5),
### I am to automate and streamline the age estimation process using the predictive analytics

In [15]:
import pandas as pd

# DATA OVERVIEW AND EXPLANATORY ANALYSIS

In [18]:
abalone = pd.read_csv('abalone.data', header=None)

In [19]:
abalone.head()

,0,1,2,3,4,5,6,7,8
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [20]:
abalone.columns = ['Sex', 'Length', 'Diameter', 'Height'
                   ,'Whole weight', 'Shucked weight', 'Viscera weight'
                   ,'Shell weight', 'Rings' ]

In [21]:
abalone.head(3)

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.15,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.07,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.21,9


In [23]:
abalone.dtypes

Sex                   str
Length            float64
Diameter          float64
Height            float64
Whole weight      float64
Shucked weight    float64
Viscera weight    float64
Shell weight      float64
Rings               int64
dtype: object

In [24]:
# convert Sex from string into category
abalone['Sex'] = abalone['Sex'].astype('category')

In [25]:
abalone.dtypes

Sex               category
Length             float64
Diameter           float64
Height             float64
Whole weight       float64
Shucked weight     float64
Viscera weight     float64
Shell weight       float64
Rings                int64
dtype: object

## The dataset utilized is the classic abalone dataset, consisting of physical attributes and the target variable Rings

## Key Dataset Attributes:
*Categorical Features*: 
Sex (Male [M], Female[F] and Infant[I])
*Continus Features*:
Length, Diameter, Height, Whole weight, Shucked weight, Viscera weight, shell weight
Target variable is Rings

In [27]:
abalone = pd.get_dummies(abalone, columns=['Sex'], dtype=int)

In [30]:
correlation_matrix = abalone.corr()
correlation_with_target = correlation_matrix['Rings'].sort_values(ascending=False)
print('Correlation of features with Rings: \n', correlation_with_target)

Correlation of features with Rings: 
 Rings             1.000000
Shell weight      0.627574
Diameter          0.574660
Height            0.557467
Length            0.556720
Whole weight      0.540390
Viscera weight    0.503819
Shucked weight    0.420884
Sex_F             0.250279
Sex_M             0.181831
Sex_I            -0.436063
Name: Rings, dtype: float64


### Correlation Analysis
Based on the initial exploratory data analysis, the continuous variable exhibits strong postive correlation with the target variable "Rings"
Among the categorical variable the SEX_I shows strong negative correlation

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [33]:
X = abalone.drop('Rings', axis=1) # predictor variable
y = abalone['Rings'] # response variable

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

In [37]:
estimators = [
    ('lr', LinearRegression()),
    ('dt', DecisionTreeRegressor(random_state=42)),
    ('rf', RandomForestRegressor(n_estimators=10, random_state=42))
]

In [40]:
models['Stacking Regressor'] = StackingRegressor(estimators=estimators, final_estimator=LinearRegression())

# PREDICTIVE MODELLING STRATEGY
#### To ensure robust prediction mechanism, i tested and evaluated four distinct algorithmic approaches, rangin from foundational to advance ensemble methods:
##### Linear Regression: Serve as baseline model to capture direct linear relationship between physical dimension and age
##### Decision Tree Regressor: A non_linear model capable of capturing complex decision boundaries though prone to overfitting
##### Random Forest Regressor: An ensemble method utilising multiple decision tree to reduce variance and improve predictive accuracy
#### Stacking Regressor: An advanced meta-ensemble technique that combine predictions of the linear regression, decision tree and random forest models,
#### using a finak linear regression estimators to compute the ultimate prediction

In [42]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MSE': mse, 'R2': r2}

In [43]:
print(results)

{'Linear Regression': {'MSE': 4.891232447128561, 'R2': 0.548162813788928}, 'Decision Tree': {'MSE': 9.21531100478469, 'R2': 0.14871757998206636}, 'Random Forest': {'MSE': 5.089339114832536, 'R2': 0.5298623219859746}, 'Stacking Regressor': {'MSE': 4.679658338698806, 'R2': 0.5677073868308691}}


# Model evaluation and results
##### The models were evaluated on the test set usign Mean Squared Error and R_Squared

Model                 Mean_square_error  R2_score   Interpretation
Decision Tree         9.21531            0.14871     Failed to generalize
Linear Regression     4.89123            0.54816     underfit: Unable to capture non linear patter
Random Forest         5.089339           0.52986     Strong performance
Stacking Regressor    4.679658           0.56770     Best performance

# Conclusion
##### The project successfully demonstrates that the age of an abalone (Rings) can be predicted using standard physical measurements, circumventing the need for the tedious microscopic counting method. 
##### The Stacking Regressor and Random Forest Regressor vastly outperformed baseline linear models, indicating that the relationship between an abalone's physical size/weight and its age is highly non-linear. 
##### The Stacking approach proved to be the most optimal, effectively minimizing the Mean Squared Error by leveraging the combined architectural strengths of multiple algorithms.

# Recommendation for future work:
To deploy this model into a production environment for marine biologists, 
I recommend the following subsequent steps:
* Hyperparameter Tuning: Utilize GridSearchCV or RandomizedSearchCV to fine-tune the tree depth and estimator counts in the Random Forest and Stacking models.
* Feature Engineering: Generate polynomial features (e.g., calculating an approximate "Volume" using Length $\times$ Diameter $\times$ Height) to give the models more direct density metrics.
* Data Scaling: Apply StandardScaler or MinMaxScaler to the continuous features (like weights and lengths) to optimize the convergence of the base models in the stacking architecture.